In [1]:
#使用するライブラリをimport
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb

#データセットの読み込み
path = '../data/raw/Milling_Tool_Dataset.csv'
df = pd.read_csv(path)

#訓練用、テスト用、検証用にデータを分割
X = df.drop(["cutting_speed", "feed_rate", "material_hardness", "tool_wear", "RUL"], axis=1)
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#2つのモデルを比較
models = {
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, scoring="neg_mean_absolute_error", cv=5)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

RandomForest: -1.9921 (+/- 0.1450)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000177 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 784, number of used features: 5
[LightGBM] [Info] Start training from score 6.977890
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

RandamForestとLightGBMのモデルを比較。

RandamForestのほうが評価が良いが、LightGBMのほうがばらつきが小さい。

両モデルにハイパーパラメータ調整を行う。

In [2]:
#RandomForestのハイパーパラメータ調整
from sklearn.model_selection import GridSearchCV

search_params = {
    'n_estimators'      : [20, 50, 100, 300],
    'random_state'      : [42],
    'n_jobs'            : [-1],
    'min_samples_split' : [5, 10, 25, 50, 100],
    'max_depth'         : [5, 10, 25, 50, 100]
}

cv = GridSearchCV(RandomForestRegressor(),search_params,verbose=2)

cv.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_sta

,estimator,RandomForestRegressor()
,param_grid,"{'max_depth': [5, 10, ...], 'min_samples_split': [5, 10, ...], 'n_estimators': [20, 50, ...], 'n_jobs': [-1], ...}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


In [3]:
#調整済みモデルで予測
RFR_model=cv.best_estimator_
train_predict = RFR_model.predict(X_train)
test_predict = RFR_model.predict(X_test)

#評価
result_list=[]
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"RandomForest",
    "dataset":"raw_data",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"model:{result['model']}, dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

model:RandomForest, dataset:raw_data
MAE_train:1.674, MAE_test:1.861
R^2_train:0.738, R^2_test:0.638


In [4]:
#LightGBMのハイパーパラメータ調整
params = {
    'objective': 'regression',
    'metric': "mae",
    'num_leaves': 31,         # default = 31,
    'learning_rate': 0.1,    # default = 0.1
    'feature_fraction': 1.0,  # default = 1.0
    'bagging_freq': 0,        # default = 0
    'bagging_fraction': 1.0,  # default = 1.0
    'random_state': 0,        # default = None
}
num_round = 100

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

import optuna
import optuna.integration.lightgbm as lgb_tuner

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

LGBM_model = model.get_best_booster()

[I 2025-11-05 14:58:21,607] A new study created in memory with name: no-name-8a798831-b436-470b-8ded-08579ab17b0d
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000162 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.924412:  14%|8     | 1/7 [00:00<00:01,  3.38it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000079 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.924412:  29%|#7    | 2/7 [00:00<00:01,  3.35it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000101 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.829313:  80%|#### | 4/5 [00:00<00:00, 33.78it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000088 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training fro

min_child_samples, val_score: 1.829313: 100%|#####| 5/5 [00:00<00:00, 33.61it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 4, 'bagging_fraction': 0.7400943912721216, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.004904863030025827, 'lambda_l2': 0.9101195191398808, 'min_child_samples': 5}


In [5]:
train_predict = LGBM_model.predict(X_train)
test_predict = LGBM_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"model:{result['model']}, dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

model:LightGBM, dataset:raw_data
MAE_train:1.800, MAE_test:1.829
R^2_train:0.691, R^2_test:0.640


パラメータ調整済みモデルは、どちらもR^2(決定係数)=0.63〜0.65, MAE(平均絶対誤差)=1.8〜1.9とあまり変わらない。

計算負荷が低く、ばらつきの少なかったLightGBMを使用して以後の分析を続行する。

In [6]:
#各特徴量のモデル作成時の寄与とモデルの予測精度への寄与を評価
from sklearn.inspection import permutation_importance
regressor_model = lgb.LGBMRegressor()
regressor_model._Booster = LGBM_model
regressor_model._n_features = X_test.shape[1]
regressor_model.fitted_ = True
results = permutation_importance(regressor_model, X_test, y_test, n_repeats=10, random_state=42)

importances_df=pd.DataFrame(zip(X.columns, LGBM_model.feature_importance(importance_type='gain'), results['importances'].mean(axis=1)), columns=['features', 'feature_importance', 'permutation_importance'])

#グラフ化
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(importances_df, x='features', y='feature_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Feature_importance')
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(importances_df, x='features', y='permutation_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Permutation_importance')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/lightgbm_feature_importances_histplot.png", format="png")
plt.close()

importances_df

,features,feature_importance,permutation_importance
0,vibration_x,1388.656189,0.022653
1,vibration_y,610.957596,0.003641
2,vibration_z,71.529200,0.001859
3,acoustic_rms,38548.496910,0.899614
4,spindle_load,6029.536543,0.071575


特徴量の重要度評価では、2つの指標はどちらも同じ傾向。"acoustic_rms"の寄与度が非常に大きく評価されていて、続いて"spindle_load", "vibration_x"となっている。これは、eda.ipynbで確認した各センサデータの"tool_wear"との相関係数の値とも矛盾しない。

次に、複数の加工データから移動平均を取ったデータセットでトライする。

In [7]:
#20行のデータの移動平均を使用したデータでトライ
path_avg20 = '../data/processed/mv_avg_20.csv'
df = pd.read_csv(path_avg20)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg20_model = model.get_best_booster()

[I 2025-11-05 14:58:29,989] A new study created in memory with name: no-name-dd06cf5f-8d87-431b-8cd2-bc115ec82e8a
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000135 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.271400:  14%|8     | 1/7 [00:00<00:01,  3.26it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000070 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.271400:  29%|#7    | 2/7 [00:00<00:01,  3.25it/s]

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000148 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 0.225830:  80%|#### | 4/5 [00:02<00:00,  1.60it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 0.225830: 100%|#####| 5/5 [00:02<00:00,  1.92it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [8]:
train_predict = avg20_model.predict(X_train)
test_predict = avg20_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_20",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

dataset:mv_avg_20
MAE_train:0.007, MAE_test:0.226
R^2_train:1.000, R^2_test:0.991


In [9]:
#10行のデータの移動平均を使用したデータでトライ
path_avg10 = '../data/processed/mv_avg_10.csv'
df = pd.read_csv(path_avg10)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg10_model = model.get_best_booster()

[I 2025-11-05 14:58:52,030] A new study created in memory with name: no-name-62e5b00a-8ded-429d-882a-0316962acde3
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000151 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.576977:  14%|8     | 1/7 [00:00<00:01,  3.37it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000093 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.576977:  29%|#7    | 2/7 [00:00<00:01,  3.32it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000073 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 0.558597:  80%|#### | 4/5 [00:00<00:00,  4.76it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 0.558597: 100%|#####| 5/5 [00:01<00:00,  4.76it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 31, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 0, 'bagging_fraction': 1.0, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 1.279354502508976e-08, 'lambda_l2': 0.003093686428960341, 'min_child_samples': 5}


In [10]:
train_predict = avg10_model.predict(X_train)
test_predict = avg10_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_10",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

dataset:mv_avg_10
MAE_train:0.150, MAE_test:0.559
R^2_train:0.998, R^2_test:0.962


In [11]:
#5行のデータの移動平均を使用したデータでトライ
path_avg5 = '../data/processed/mv_avg_5.csv'
df = pd.read_csv(path_avg5)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg5_model = model.get_best_booster()

[I 2025-11-05 14:59:10,217] A new study created in memory with name: no-name-a7721fe2-a87e-4519-8bcc-5fe109c74cf5
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000133 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.922167:  14%|8     | 1/7 [00:00<00:01,  3.62it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.922167:  29%|#7    | 2/7 [00:00<00:01,  3.58it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 0.917595:  80%|#### | 4/5 [00:00<00:00,  4.41it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000104 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 0.917595: 100%|#####| 5/5 [00:01<00:00,  4.44it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [12]:
train_predict = avg5_model.predict(X_train)
test_predict = avg5_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_5",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

dataset:mv_avg_5
MAE_train:0.392, MAE_test:0.918
R^2_train:0.985, R^2_test:0.908


In [13]:
#2行のデータの移動平均を使用したデータでトライ
path_avg2 = '../data/processed/mv_avg_2.csv'
df = pd.read_csv(path_avg2)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg2_model = model.get_best_booster()

[I 2025-11-05 14:59:29,777] A new study created in memory with name: no-name-38433f28-d232-48ab-af89-7d66edb462aa
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000144 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.581950:  14%|8     | 1/7 [00:00<00:01,  3.31it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000075 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.581950:  29%|#7    | 2/7 [00:00<00:01,  3.32it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000103 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000071 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000102 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5


min_child_samples, val_score: 1.512842:  80%|#### | 4/5 [00:00<00:00, 25.12it/s]

[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000125 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.512842: 100%|#####| 5/5 [00:00<00:00, 24.40it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 3, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 7, 'bagging_fraction': 0.9898882087115335, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.0009934294826405247, 'lambda_l2': 0.9834881107044738, 'min_child_samples': 20}


In [14]:
train_predict = avg2_model.predict(X_train)
test_predict = avg2_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_2",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

dataset:mv_avg_2
MAE_train:1.248, MAE_test:1.513
R^2_train:0.850, R^2_test:0.754


In [15]:
#MAEとR^2の推移を表示
result_df = pd.DataFrame(result_list)
lgb_results=result_df[result_df["model"]=="LightGBM"]

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(lgb_results, x='dataset', y='MAE_test', errorbar=None)
plt.title('MAE')
plt.xticks(rotation=45)
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(lgb_results, x='dataset', y='R^2_test', errorbar=None)
plt.xticks(rotation=45)
plt.title('R^2')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/raw_mvavg_result_histplot.png", format="png")
plt.close()

lgb_results[["dataset", "MAE_test", "R^2_test"]]

,dataset,MAE_test,R^2_test
1,raw_data,1.829313,0.640487
2,mv_avg_20,0.225830,0.990745
3,mv_avg_10,0.558597,0.961693
4,mv_avg_5,0.917595,0.907897
5,mv_avg_2,1.512842,0.753801


過去のデータ数を多く使用したデータほど予測精度が良い(MAEが小さくR^2が大きい)モデルが作成できた。

次に各センサのデータに"pl_vib_vec"を追加したデータセットでトライする。

In [16]:
path_pvec = '../data/processed/add_plane_vec.csv'
df = pd.read_csv(path_pvec)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

add_pvec_model = model.get_best_booster()

[I 2025-11-05 14:59:38,217] A new study created in memory with name: no-name-9d9c162f-d833-4da8-90aa-38b40942ab37
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000168 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.966486:  14%|8     | 1/7 [00:00<00:01,  3.58it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.950997:  29%|#7    | 2/7 [00:00<00:01,  3.42it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000149 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000092 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.834035:  60%|###  | 3/5 [00:00<00:00, 18.29it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.834035: 100%|#####| 5/5 [00:00<00:00, 18.78it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000113 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 4, 'learning_rate': 0.1, 'feature_fraction': 0.8999999999999999, 'bagging_freq': 1, 'bagging_fraction': 0.9030454357272278, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 1.0011678239049503e-08, 'lambda_l2': 7.563451957844805, 'min_child_samples': 20}


In [17]:
train_predict = add_pvec_model.predict(X_train)
test_predict = add_pvec_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"add_plane_vec",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
result_list.append(result)

dataset:add_plane_vec
MAE_train:1.675, MAE_test:1.834
R^2_train:0.730, R^2_test:0.639


In [18]:
regressor_model = lgb.LGBMRegressor()
regressor_model._Booster = add_pvec_model
regressor_model._n_features = X_test.shape[1]
regressor_model.fitted_ = True
results_2 = permutation_importance(regressor_model, X_test, y_test, n_repeats=10, random_state=42)

importances_add_pvec_df=pd.DataFrame(zip(X.columns, add_pvec_model.feature_importance(importance_type='gain'), results_2['importances'].mean(axis=1)), columns=['features', 'feature_importance', 'permutation_importance'])

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(importances_add_pvec_df, x='features', y='feature_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Feature_importance')
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(importances_add_pvec_df, x='features', y='permutation_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Permutation_importance')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/add_pvec_importances_histplot.png", format="png")
plt.close()

importances_add_pvec_df

,features,feature_importance,permutation_importance
0,pl_vib_vec,2541.334183,0.017237
1,vibration_x,1197.302445,0.006594
2,vibration_y,296.320590,0.002518
3,vibration_z,419.277924,0.004112
4,acoustic_rms,47174.189667,0.894053
5,spindle_load,6877.846788,0.078962


In [19]:
#MAEとR^2の推移を表示
result_df = pd.DataFrame(result_list)
con_df = pd.concat([result_df[(result_df["model"]=="LightGBM")&(result_df["dataset"]=="raw_data")],
           result_df[result_df["dataset"]=="add_plane_vec"]])

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(con_df, x='dataset', y='MAE_test', errorbar=None)
plt.title('MAE')
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(con_df, x='dataset', y='R^2_test', errorbar=None)
plt.title('R^2')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/raw_add_pvec_result_histplot.png", format="png")
plt.close()

con_df[["dataset", "MAE_test", "R^2_test"]]

,dataset,MAE_test,R^2_test
1,raw_data,1.829313,0.640487
6,add_plane_vec,1.834035,0.638516


生データと"pl_vib_vec"を追加したデータセットでMAEもR^2もほとんど変わらなかった。特徴量の重要度評価からも、追加した特徴量はあまりモデルの予測精度には寄与しなかったことが分かる。

複数回の加工のデータの移動平均を使用することでモデルの予測精度を高めることができたが、１回の加工単位ではセンサー値のブレが大きく、特徴量化が難しいと考える。

続いて、必要なセンサを確認するため、重要度評価が低いセンサのデータを削減してモデルを再度作成する。使用するデータセットは、生データと作成した中で最も予測精度の良いモデルを作成できた、20回の加工データの移動平均をとったデータセット。

In [20]:
#生データから重要度評価が最も低い特徴量を削除して再評価
path = '../data/raw/Milling_Tool_Dataset.csv'
df = pd.read_csv(path)

X = df[["vibration_x", "vibration_y", "acoustic_rms", "spindle_load"]]
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

z_del_model = model.get_best_booster()

[I 2025-11-05 14:59:47,819] A new study created in memory with name: no-name-b4a2c779-7458-4e58-acd0-b45db0b77ec2
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000187 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.990633:  14%|8     | 1/7 [00:00<00:01,  3.52it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000103 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.990633:  29%|#7    | 2/7 [00:00<00:01,  3.65it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000069 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.843530:  80%|#### | 4/5 [00:00<00:00, 30.97it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000072 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000101 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training fro

min_child_samples, val_score: 1.843530: 100%|#####| 5/5 [00:00<00:00, 30.82it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 2, 'bagging_fraction': 0.8122699238897743, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.13646593958080025, 'lambda_l2': 1.1164696868156605e-05, 'min_child_samples': 5}


In [21]:
train_predict = z_del_model.predict(X_train)
test_predict = z_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data_del_z",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

full_data_MAE=result_df[(result_df["model"]=="LightGBM")&(result_df["dataset"]=="raw_data")]["MAE_test"][1]
print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
print(f"MAE_change_rate:{(result['MAE_test']-full_data_MAE)/full_data_MAE*100:.3f}")
result_list.append(result)

dataset:raw_data_del_z
MAE_train:1.797, MAE_test:1.844
R^2_train:0.693, R^2_test:0.638
MAE_change_rate:0.777


In [22]:
#MAEが元データの5%下がるまで特徴量の削除を続行
X = df[["vibration_x", "acoustic_rms", "spindle_load"]]
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

yz_del_model = model.get_best_booster()

[I 2025-11-05 14:59:56,115] A new study created in memory with name: no-name-7c8efd3a-1a14-4625-9464-ee766460f7b9
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000056 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.963070:  14%|8     | 1/7 [00:00<00:01,  3.43it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000070 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.963070:  29%|#7    | 2/7 [00:00<00:01,  3.36it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.845637:  80%|#### | 4/5 [00:00<00:00, 32.78it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000113 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000073 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.845637: 100%|#####| 5/5 [00:00<00:00, 32.77it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 3, 'bagging_fraction': 0.8765936548286843, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.018632903933684837, 'lambda_l2': 2.4466954544486247e-07, 'min_child_samples': 20}


In [23]:
train_predict = yz_del_model.predict(X_train)
test_predict = yz_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data_del_yz",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
print(f"MAE_change_rate:{(result['MAE_test']-full_data_MAE)/full_data_MAE*100:.3f}")
result_list.append(result)

dataset:raw_data_del_yz
MAE_train:1.820, MAE_test:1.846
R^2_train:0.688, R^2_test:0.635
MAE_change_rate:0.892


In [24]:
X = df[["acoustic_rms", "spindle_load"]]
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

vib_del_model = model.get_best_booster()

[I 2025-11-05 15:00:03,956] A new study created in memory with name: no-name-814b6b6e-02c5-4267-9d03-57004ccf22f3
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.959869:  14%|8     | 1/7 [00:00<00:01,  3.39it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000107 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.959869:  29%|#7    | 2/7 [00:00<00:01,  3.35it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000067 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000093 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.859718:  80%|#### | 4/5 [00:00<00:00, 33.59it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000067 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.859718: 100%|#####| 5/5 [00:00<00:00, 33.41it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 2, 'bagging_fraction': 0.5986052925633432, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 7.990637613568693, 'lambda_l2': 0.0009327368847797601, 'min_child_samples': 5}


In [25]:
train_predict = vib_del_model.predict(X_train)
test_predict = vib_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data_del_vib",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
print(f"MAE_change_rate:{(result['MAE_test']-full_data_MAE)/full_data_MAE*100:.3f}")
result_list.append(result)

dataset:raw_data_del_vib
MAE_train:1.874, MAE_test:1.860
R^2_train:0.672, R^2_test:0.634
MAE_change_rate:1.662


In [26]:
from sklearn.linear_model import LinearRegression
X = df[['acoustic_rms']]
y = df['tool_wear']

model = LinearRegression()
model.fit(X, y)
y_predict = model.predict(X)

result = {
    "model":"Lenear",
    "dataset":"acoustic_rms_and_tool_wear",
    "MAE":mean_absolute_error(y, y_predict),
    "R^2":r2_score(y, y_predict),
}

result_list.append(result)

print(f"dataset:{result['dataset']}")
print(f"MAE: {result['MAE']:.3f}")
print(f"R²: {result['R^2']:.3f}")
print(f"MAE_change_rate:{(result['MAE']-full_data_MAE)/full_data_MAE*100:.3f}")
print(f"回帰式: tool_wear = {model.coef_[0]:.4f} * acoustic_rms + {model.intercept_:.4f}")

plt.scatter(X, y, alpha=0.4, label='Actual')
plt.plot(X, y_predict, color='red', label='Regression line')
plt.xlabel('acoustic_rms')
plt.ylabel('tool_wear')
plt.legend()
plt.savefig("../outputs/figures/modeling/raw_acoustic_lineplot.png", format="png")
plt.close()

dataset:acoustic_rms_and_tool_wear
MAE: 2.105
R²: 0.587
MAE_change_rate:15.071
回帰式: tool_wear = 40.3962 * acoustic_rms + -17.1735


生データから振動データ(vibration_x, vibration_y, vibration_z)を削除しても、予測精度はほとんど変わらなかった。

さらに"spindle_load"を削除し、"acoustic_rms"のみを使用して単回帰分析を行った。この分析の評価は R^2=0.587, MAE=2.105であり、"acoustic_rms"のみでもある程度"tool_wear"を説明できている。

以上から、音響データ(acoustic_rms)が振動データやスピンドル負荷(spindle_load)データを内包している可能性が考えられる。

続いて、20回分の加工データの移動平均を使用したデータセットでも、振動データと"spindle_load"を除去してモデルを作成し、精度の変化を確認する。

In [27]:
path_avg20 = '../data/processed/mv_avg_20.csv'
df = pd.read_csv(path_avg20)

X = df[["mv_avg_ar", "mv_avg_sl"]]
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg20_vib_del_model = model.get_best_booster()

[I 2025-11-05 15:00:12,268] A new study created in memory with name: no-name-b1bec30a-2c59-469b-bcd2-ea7fc56dacd2
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000175 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.403093:  14%|8     | 1/7 [00:00<00:01,  3.60it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.403093:  29%|#7    | 2/7 [00:00<00:01,  3.66it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000072 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

min_child_samples, val_score: 0.399727:  80%|#### | 4/5 [00:00<00:00,  3.97it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

min_child_samples, val_score: 0.399727: 100%|#####| 5/5 [00:01<00:00,  4.39it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 41, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 3, 'bagging_fraction': 0.8348405717920351, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 1.1115480349521012e-08, 'lambda_l2': 5.9308186435169326, 'min_child_samples': 20}


In [28]:
train_predict = avg20_vib_del_model.predict(X_train)
test_predict = avg20_vib_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg20_del_vib",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

avg20_full_data_MAE = result_df[result_df["dataset"]=="mv_avg_20"]['MAE_test'][2]
print(f"dataset:{result['dataset']}")
print(f"MAE_train:{result['MAE_train']:.3f}, MAE_test:{result['MAE_test']:.3f}")
print(f"R^2_train:{result['R^2_train']:.3f}, R^2_test:{result['R^2_test']:.3f}")
print(f"MAE_change_rate:{(result['MAE_test']-avg20_full_data_MAE)/avg20_full_data_MAE*100:.3f}")
result_list.append(result)

dataset:mv_avg20_del_vib
MAE_train:0.289, MAE_test:0.400
R^2_train:0.991, R^2_test:0.982
MAE_change_rate:77.004


In [29]:
X = df[['mv_avg_ar']]
y = df['tool_wear']

model = LinearRegression()
model.fit(X, y)
y_predict = model.predict(X)

result = {
    "model":"Lenear",
    "dataset":"mv_avg_ar_and_tool_wear",
    "MAE":mean_absolute_error(y, y_predict),
    "R^2":r2_score(y, y_predict),
}
result_list.append(result)

print(f"dataset:{result['dataset']}")
print(f"MAE: {result['MAE']:.3f}")
print(f"R²: {result['R^2']:.3f}")
print(f"MAE_change_rate:{(result['MAE']-avg20_full_data_MAE)/avg20_full_data_MAE*100:.3f}")
print(f"回帰式: tool_wear = {model.coef_[0]:.4f} * acoustic_rms + {model.intercept_:.4f}")

plt.scatter(X, y, alpha=0.4, label='Actual')
plt.plot(X, y_predict, color='red', label='Regression line')
plt.xlabel('acoustic_rms')
plt.ylabel('tool_wear')
plt.legend()
plt.savefig("../outputs/figures/modeling/mv_avg20_acoustic_lineplot.png", format="png")
plt.close()

dataset:mv_avg_ar_and_tool_wear
MAE: 0.572
R²: 0.971
MAE_change_rate:153.303
回帰式: tool_wear = 66.4680 * acoustic_rms + -32.6708


20回分の加工データの移動平均を取ったデータセットから振動データを削除し、モデルを作成した。

R^2の低下とMAEの上昇から、振動データありのモデルよりも予測精度が下がっていることが分かる。

次に"acoustic_rms"のみで単回帰分析を行うと、さらなる精度の低下がおきた。

このことから、高精度な予測をしようとすると全てのセンサが必要なことが確認できた。

ただし、振動データと"spindle_load"を両方削除したデータセットから作成されたモデルでもR^2は非常に高く、高い予測精度を求めなければ、音響センサ(acoustic_rms)のみでも十分予測精度が高いモデルが構築可能。

In [30]:
#各モデルの結果を保存する
result_df=pd.DataFrame(result_list)
result_df.to_csv('../outputs/data/metrics_summury.csv', encoding="utf-8-sig", index=False)
result_df

,model,dataset,MAE_train,MAE_test,R^2_train,R^2_test,MAE,R^2
0,RandomForest,raw_data,1.673694,1.860607,0.738418,0.638382,NaN,NaN
1,LightGBM,raw_data,1.799813,1.829313,0.691366,0.640487,NaN,NaN
2,LightGBM,mv_avg_20,0.007325,0.225830,0.999977,0.990745,NaN,NaN
3,LightGBM,mv_avg_10,0.150351,0.558597,0.997595,0.961693,NaN,NaN
4,LightGBM,mv_avg_5,0.391799,0.917595,0.985096,0.907897,NaN,NaN
5,LightGBM,mv_avg_2,1.248440,1.512842,0.850049,0.753801,NaN,NaN
6,LightGBM,add_plane_vec,1.675362,1.834035,0.730427,0.638516,NaN,NaN
7,LightGBM,raw_data_del_z,1.796833,1.843530,0.692792,0.638442,NaN,NaN
8,LightGBM,raw_data_del_yz,1.820429,1.845637,0.688169,0.635196,NaN,NaN
9,LightGBM,raw_data_del_vib,1.874467,1.859718,0.672430,0.633627,NaN,NaN


まとめ

工具摩耗値の予測を高精度にしようとするなら複数の加工データと全てのセンサのデータが必要である。 


加工データの数について、求められる加工精度にもよるため一概には言えないが、データ数5以上であればR^2が0.9以上の非常に予測精度が良いモデルが作成可能。

１回の加工データからの予測ではMAEは1.8〜1.9程度で、振動センサのデータを無くしてもほとんどモデルの予測精度には影響しない。

必要なセンサについて、音響センサだけでも20回分の加工データがあれば十分に精度の良いモデルは作成可能であり、高精度が必要でなければ音響センサのみでもよい。